In [29]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense

In [30]:
import pandas as pd

In [31]:
df = pd.read_csv("/content/diabetes.csv")

In [32]:
# checking the data
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [33]:
# separating features and target
X = df.drop(["Outcome"],axis=1)
y = df[["Outcome"]]

In [34]:
from sklearn.preprocessing import StandardScaler

In [35]:
X_scaled = StandardScaler().fit_transform(X)

In [36]:
from sklearn.model_selection import train_test_split

In [37]:
X_train,X_test,y_train,y_test = train_test_split(X_scaled,y,test_size=0.2,random_state=21)

In [38]:
X_train.shape

(614, 8)

In [39]:
model = Sequential()
model.add(Dense(32,activation="relu",input_dim=8))
model.add(Dense(1,activation="sigmoid"))
model.compile(optimizer="Adam",loss="binary_crossentropy",metrics=["accuracy"])

In [40]:
model.fit(X_train,y_train,batch_size=32,epochs=100,validation_data=(X_test,y_test))

Epoch 1/100
20/20 [==============================] - 1s 11ms/step - loss: 0.7321 - accuracy: 0.4853 - val_loss: 0.7096 - val_accuracy: 0.5390
Epoch 2/100
20/20 [==============================] - 0s 3ms/step - loss: 0.6721 - accuracy: 0.5700 - val_loss: 0.6633 - val_accuracy: 0.5909
Epoch 3/100
20/20 [==============================] - 0s 5ms/step - loss: 0.6259 - accuracy: 0.6466 - val_loss: 0.6307 - val_accuracy: 0.6039
Epoch 4/100
20/20 [==============================] - 0s 4ms/step - loss: 0.5907 - accuracy: 0.6906 - val_loss: 0.6047 - val_accuracy: 0.6429
Epoch 5/100
20/20 [==============================] - 0s 6ms/step - loss: 0.5623 - accuracy: 0.7117 - val_loss: 0.5828 - val_accuracy: 0.6558
Epoch 6/100
20/20 [==============================] - 0s 6ms/step - loss: 0.5392 - accuracy: 0.7296 - val_loss: 0.5658 - val_accuracy: 0.6688
Epoch 7/100
20/20 [==============================] - 0s 5ms/step - loss: 0.5207 - accuracy: 0.7443 - val_loss: 0.5536 - val_accuracy: 0.6883
Epoch 8/100


In [41]:
!pip install keras-tuner

In [42]:
import keras_tuner as kt

# Hyperparameter Tuning
- The following code will search for best optimizers among adam,rmsprop, sgd and adadelta.

In [43]:
# this function is necessary for hyperparameter tuning
def build_model(hp):
  model = Sequential()
  model.add(Dense(32,activation="relu",input_dim=8))
  model.add(Dense(1,activation="sigmoid"))

  optimizers = hp.Choice("optimizer",values=["adam","rmsprop","sgd","adadelta"])
  model.compile(optimizer=optimizers,loss="binary_crossentropy",metrics=["accuracy"])
  return model

In [44]:
# tk.RandomSearch first argument is function we made to find the best hyperparameter (optimizer in above code)
# second parameter is thing we want to increase (validation data accuracy in our case)
# max_trials number of trails to do
tuner = kt.RandomSearch(build_model,objective="val_accuracy",max_trials=5)

Reloading Tuner from ./untitled_project/tuner0.json


In [45]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

In [48]:
tuner.results_summary()

Results summary
Results in ./untitled_project
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 3 summary
Hyperparameters:
optimizer: adam
Score: 0.6948052048683167

Trial 1 summary
Hyperparameters:
optimizer: rmsprop
Score: 0.6753246784210205

Trial 0 summary
Hyperparameters:
optimizer: sgd
Score: 0.6753246784210205

Trial 2 summary
Hyperparameters:
optimizer: adadelta
Score: 0.38311687111854553


In [49]:
# geting the best parameter
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'adam'}

In [51]:
# getting the best model
best_model = tuner.get_best_models(num_models=1)[0]

In [52]:
best_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 32)                288       
                                                                 
 dense_1 (Dense)             (None, 1)                 33        
                                                                 
Total params: 321 (1.25 KB)
Trainable params: 321 (1.25 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [54]:
# using the best model
# initial_epoch is set 6 as our model has already run 5 epochs above
best_model.fit(X_train,y_train,batch_size=32,epochs=100,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/100
20/20 [==============================] - 1s 12ms/step - loss: 0.5363 - accuracy: 0.7459 - val_loss: 0.5660 - val_accuracy: 0.6948
Epoch 8/100
20/20 [==============================] - 0s 4ms/step - loss: 0.5149 - accuracy: 0.7590 - val_loss: 0.5548 - val_accuracy: 0.6948
Epoch 9/100
20/20 [==============================] - 0s 4ms/step - loss: 0.4989 - accuracy: 0.7720 - val_loss: 0.5488 - val_accuracy: 0.7013
Epoch 10/100
20/20 [==============================] - 0s 4ms/step - loss: 0.4858 - accuracy: 0.7834 - val_loss: 0.5447 - val_accuracy: 0.7078
Epoch 11/100
20/20 [==============================] - 0s 4ms/step - loss: 0.4758 - accuracy: 0.7899 - val_loss: 0.5424 - val_accuracy: 0.7143
Epoch 12/100
20/20 [==============================] - 0s 4ms/step - loss: 0.4675 - accuracy: 0.7834 - val_loss: 0.5400 - val_accuracy: 0.7143
Epoch 13/100
20/20 [==============================] - 0s 3ms/step - loss: 0.4606 - accuracy: 0.7866 - val_loss: 0.5399 - val_accuracy: 0.7143
Epoch 14

# Hyperparameter tuning of Neurons
- Now, we will find what number of neurons are the best.

In [74]:
def build_model(hp):
  model = Sequential()

  # defining neuron space to search, minimum is 8 and max is 128, it will search between this numer with  a step of 8
  units = hp.Int("units",min_value = 8,max_value = 128,step=8)
  model.add(Dense(units=units,activation="relu",input_dim=8))
  model.add(Dense(1,activation="sigmoid"))
  model.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])
  return model

In [75]:
tuner2 = kt.RandomSearch(build_model,objective="val_accuracy",max_trials=5,directory="mydir2",project_name="neuron_tuning2")

In [76]:
tuner2.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 5 Complete [00h 00m 03s]
val_accuracy: 0.701298713684082

Best val_accuracy So Far: 0.7402597665786743
Total elapsed time: 00h 00m 11s


In [77]:
# getting the best number of neurons
tuner2.get_best_hyperparameters()[0].values

{'units': 80}

In [78]:
best_model2 = tuner2.get_best_models(num_models=1)[0]

In [79]:
best_model2.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 80)                720       


                                                                 
 dense_1 (Dense)             (None, 1)                 81        


Total params: 801 (3.13 KB)


Trainable params: 801 (3.13 KB)


Non-trainable params: 0 (0.00 Byte)


_________________________________________________________________


In [80]:
best_model.fit(X_train,y_train,epochs=100,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/100
20/20 [==============================] - 0s 13ms/step - loss: 0.3973 - accuracy: 0.8127 - val_loss: 0.5551 - val_accuracy: 0.7338
Epoch 8/100
20/20 [==============================] - 0s 8ms/step - loss: 0.3969 - accuracy: 0.8143 - val_loss: 0.5565 - val_accuracy: 0.7338
Epoch 9/100
20/20 [==============================] - 0s 8ms/step - loss: 0.3968 - accuracy: 0.8143 - val_loss: 0.5568 - val_accuracy: 0.7338
Epoch 10/100
20/20 [==============================] - 0s 7ms/step - loss: 0.3964 - accuracy: 0.8143 - val_loss: 0.5585 - val_accuracy: 0.7338
Epoch 11/100
20/20 [==============================] - 0s 7ms/step - loss: 0.3964 - accuracy: 0.8192 - val_loss: 0.5559 - val_accuracy: 0.7338
Epoch 12/100
20/20 [==============================] - 0s 5ms/step - loss: 0.3959 - accuracy: 0.8160 - val_loss: 0.5591 - val_accuracy: 0.7338
Epoch 13/100
20/20 [==============================] - 0s 5ms/step - loss: 0.3956 - accuracy: 0.8160 - val_loss: 0.5572 - val_accuracy: 0.7338
Epoch 14

# Hyperparameter Tuning for layers
- Now, we will search for the best number of layers.

In [84]:
def build_model(hp):

  model = Sequential()

  model.add(Dense(72,activation="relu",input_dim=8))

  # On first iteration when for loop will get 1, it will ru nonce, in next turn it recive 2 and it will run two times, next it reciev 3 and it will run 3 times adding 3 layer, so on upto 10th layer
  for i in range(hp.Int("number of layers",min_value=1,max_value=10)):
    model.add(Dense(72,activation="relu"))

  # adding the output layer outside of for loop
  model.add(Dense(1,activation="sigmoid"))

  # compiling the model
  model.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])

  return model

In [85]:
tuner3 = kt.RandomSearch(build_model,objective="val_accuracy",max_trials=3,directory="mydir",project_name="number of best layers")

In [86]:
tuner3.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 3 Complete [00h 00m 03s]
val_accuracy: 0.6623376607894897

Best val_accuracy So Far: 0.701298713684082
Total elapsed time: 00h 00m 10s


In [87]:
# number of best layers
tuner3.get_best_hyperparameters()[0].values

{'number of layers': 9}

In [88]:
# saving the best models
best_model3 = tuner3.get_best_models(num_models=1)[0]

# Now, all at once.
- We will tune multiple hyperparameters in one go.

In [89]:
from keras.layers import Dropout

In [132]:
X_train.shape

(614, 8)

In [144]:
def build_model2(hp):

  model = Sequential()

  counter = 0

  for i in range(hp.Int("Layers",min_value=1,max_value=10)):
    if counter == 0:
      model.add(Dense(hp.Int("neuron" + str(i),min_value=8,max_value=128,step=8),activation=hp.Choice("acitvation" + str(i),values=["relu","tanh",]),input_dim=8))
      model.add(Dropout(hp.Choice("dropout" + str(i),values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))
    else:
      model.add(Dense(hp.Int("neuron" + str(i),min_value=8,max_value=128,step=8),activation=hp.Choice("activation" + str(i),values=["relu","tanh"])))
      model.add(Dropout(hp.Choice("dropout" + str(i),values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))
    counter+=1

  model.add(Dense(1,activation="sigmoid"))
  model.compile(optimizer=hp.Choice("Optimizer" + str(i),values = ["adam","sgd","adadelta","rmsprop"]),loss="binary_crossentropy",metrics=["accuracy"])

  return model


In [145]:
tuner4 = kt.RandomSearch(build_model2,objective="val_accuracy",max_trials=3,directory="mydir",project_name="all at once42222")


In [146]:
tuner4.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 3 Complete [00h 00m 04s]
val_accuracy: 0.6298701167106628

Best val_accuracy So Far: 0.7077922224998474
Total elapsed time: 00h 00m 11s


In [147]:
tuner4.get_best_hyperparameters()[0].values

{'Layers': 6,
 'neuron0': 16,
 'acitvation0': 'tanh',
 'dropout0': 0.5,
 'Optimizer0': 'adadelta',
 'neuron1': 16,
 'activation1': 'relu',
 'dropout1': 0.5,
 'neuron2': 120,
 'activation2': 'tanh',
 'dropout2': 0.3,
 'neuron3': 120,
 'activation3': 'tanh',
 'dropout3': 0.2,
 'neuron4': 56,
 'activation4': 'relu',
 'dropout4': 0.2,
 'Optimizer4': 'sgd',
 'neuron5': 8,
 'activation5': 'relu',
 'dropout5': 0.1,
 'Optimizer5': 'adam'}

In [148]:
best_model = tuner4.get_best_models(num_models=1)[0]

In [149]:
best_model.fit(X_train,y_train,epochs=200,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/200
20/20 [==============================] - 3s 22ms/step - loss: 0.5494 - accuracy: 0.6873 - val_loss: 0.5645 - val_accuracy: 0.6688
Epoch 8/200
20/20 [==============================] - 0s 8ms/step - loss: 0.5450 - accuracy: 0.7248 - val_loss: 0.5577 - val_accuracy: 0.6818
Epoch 9/200
20/20 [==============================] - 0s 7ms/step - loss: 0.5253 - accuracy: 0.7182 - val_loss: 0.5572 - val_accuracy: 0.6818
Epoch 10/200
20/20 [==============================] - 0s 7ms/step - loss: 0.5501 - accuracy: 0.7182 - val_loss: 0.5505 - val_accuracy: 0.7143
Epoch 11/200
20/20 [==============================] - 0s 7ms/step - loss: 0.5595 - accuracy: 0.7166 - val_loss: 0.5505 - val_accuracy: 0.7078
Epoch 12/200
20/20 [==============================] - 0s 7ms/step - loss: 0.5380 - accuracy: 0.7378 - val_loss: 0.5365 - val_accuracy: 0.7078
Epoch 13/200
20/20 [==============================] - 0s 7ms/step - loss: 0.5539 - accuracy: 0.7264 - val_loss: 0.5364 - val_accuracy: 0.7338
Epoch 14

In [150]:
best_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 16)                144       
                                                                 
 dropout (Dropout)           (None, 16)                0         
                                                                 
 dense_1 (Dense)             (None, 16)                272       
                                                                 
 dropout_1 (Dropout)         (None, 16)                0         
                                                                 
 dense_2 (Dense)             (None, 120)               2040      
                                                                 
 dropout_2 (Dropout)         (None, 120)               0         
                                                                 
 dense_3 (Dense)             (None, 120)               1